# CUDA K-means clustering
Select a T4 GPU runtime and choose **Runtime → Run all**. This notebook generates datasets from 10,000 to 1,000,000 points, benchmarks CPU and CUDA, checks their results, and downloads the execution evidence.

In [ ]:
from pathlib import Path
import subprocess
import tempfile
import sys

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy>=1.26.4,<3", "matplotlib>=3.8,<4"], check=True)
project = Path(tempfile.mkdtemp(prefix="cuda-kmeans-", dir="/content"))
print(project)


In [ ]:
files = {'README.md': "# CUDA K-Means Clustering\n\nThis capstone compares serial C++ and CUDA C++ implementations of Lloyd's K-means on five reproducible two-dimensional datasets, ranging from 10,000 to 1,000,000 points. Both use identical starting points, distance calculations, stopping rules, and inputs.\n\n## Run on Google Colab\n\nOpen the supplied CUDA_KMeans_Clustering.ipynb notebook in Colab. Select **Runtime → Change runtime type → T4 GPU**, then choose **Run all**. The notebook writes the project files into the runtime, checks for NVIDIA hardware and nvcc, runs the benchmark, and offers a download of proof-gpu.zip. It uses a fresh project directory so earlier work is preserved.\n\n## Run on Linux\n\nUse Bash, GNU Make, a C++17 compiler supported by your CUDA toolkit, and Python 3.10–3.12. GPU execution needs an NVIDIA driver and CUDA Toolkit with nvcc on PATH. The default build targets the Tesla T4. Set CUDA_ARCH to your GPU's SM target when using a different card.\n\n```bash\npython3 -m venv .venv\nsource .venv/bin/activate\npython3 -m pip install -r requirements.txt\n./run.sh gpu\n```\n\nThe CPU-only workflow uses the same dependencies and runs with `./run.sh cpu`. For a build only, use `make cpu` or `make cuda`.\n\n## Command-line interface\n\n```text\nbuild/kmeans INPUT.csv K MAX_ITERATIONS {cpu|gpu} OUTPUT_DIR [REPEATS]\n```\n\nThe CPU-only executable is `build/kmeans_cpu`. Input uses the header `x,y` and finite coordinates within ±1e100. K must be between 1 and 256 and no larger than the row count. Output includes `clusters.csv` and `centroids.csv`. Standard output reports mode, points, K, iterations, repeats, median milliseconds, and inertia.\n\n## Algorithm and measurement\n\n- The first input row seeds the first centroid. Successive seeds are the points farthest from their nearest existing seed.\n- A CUDA thread assigns one point to the nearest centroid. Threads accumulate FP64 coordinate sums and integer counts in block-local shared memory. Each block merges its occupied bins into global totals with atomics.\n- The host reads the compact totals, updates centroid means, and retains previous centers for empty clusters. The run stops when the largest centroid movement is at most 1e-6 or the iteration limit is reached. Both implementations recompute final labels from the returned centroids.\n- Timings are wall-clock medians of five runs after one warm-up per mode and size. The measured CUDA call includes device allocation, transfers, kernels, host updates and final labels. Both modes exclude CSV I/O, shared seed initialization and inertia calculation. Speedup is CPU median divided by CUDA median. Atomic summation order can change low-order digits.\n- For these seeded datasets the comparison requires identical labels, maximum centroid distance at most 1e-6, and relative inertia difference at most 1e-8.\n\n## Evidence and submission\n\nThe runner creates benchmark.csv, before-and-after and runtime plots, an execution log, the complete 10,000-point input and output, checksums for all generated inputs, and the one-million-point centroids. CUDA mode also creates comparison.csv. Full intermediate CSVs remain under output/. The archive contains the results/ directory.\n\n## Measured Tesla T4 results\n\nThese are the CPU and GPU medians from the same Google Colab session using CUDA Toolkit 13.0.\n\n| Points | CPU (ms) | GPU (ms) | CPU / GPU |\n| ---: | ---: | ---: | ---: |\n| 10,000 | 0.284 | 0.388 | 0.73× |\n| 50,000 | 1.312 | 0.773 | 1.70× |\n| 100,000 | 3.842 | 1.239 | 3.10× |\n| 500,000 | 35.365 | 5.304 | 6.67× |\n| 1,000,000 | 36.409 | 9.209 | 3.95× |\n\nAll five runs had zero label disagreements. The maximum centroid difference across the suite was 2.45e-13 and the reported inertia matched at every size. The T4 was slower at 10,000 points and faster at every larger tested size. These medians describe one Colab session and can vary with runtime load.\n\nThe accompanying proof-gpu.zip contains the locally packaged Colab evidence. The Colab export cell also offers the original runtime archive. Proof-cpu.zip contains a separate CPU-only run. The source, evidence, and narrated demonstration are available as local files.\n", 'PROJECT_DESCRIPTION.txt': "This project implements Lloyd's K-means clustering in serial C++ and CUDA C++ and compares their performance on five reproducible two-dimensional datasets ranging from 10,000 to one million points. In the CUDA implementation, each thread assigns one point to its nearest centroid. Thread blocks accumulate coordinate sums and point counts in shared memory before merging the totals, and the host uses those totals to update centroids and check convergence. Both implementations use the same starting centroids and stopping rule, and the benchmark compares median wall-clock runtime while checking labels, centroids, and inertia. The project reports transfer and allocation costs as part of the CUDA time so the comparison reflects the cost of running the clustering operation. On the Tesla T4, the packaged implementation took 9.21 ms at one million points, compared with 36.41 ms on the Colab CPU, giving a 3.95× speedup. The CPU and GPU agreed on every label across all five workloads.\n", 'Makefile': 'CXXFLAGS ?= -O3 -std=c++17 -Wall -Wextra -pedantic\nNVCC ?= nvcc\nCUDA_ARCH ?= sm_75\n\n.PHONY: cpu cuda clean\ncpu: build/kmeans_cpu\ncuda: build/kmeans\n\nbuild:\n\tmkdir -p $@\n\nbuild/kmeans_cpu: src/main.cc src/kmeans.h | build\n\t$(CXX) $(CXXFLAGS) $< -o $@\n\nbuild/kmeans: src/main.cc src/gpu.cu src/kmeans.h | build\n\t$(NVCC) -O3 -std=c++17 -arch=$(CUDA_ARCH) -DENABLE_CUDA src/main.cc src/gpu.cu -o $@\n\nclean:\n\trm -rf build\n', 'run.sh': '#!/usr/bin/env bash\nset -euo pipefail\ncd "$(dirname "$0")"\nmode=${1:-gpu}\ncase "$mode" in cpu|gpu) ;; *) echo "Usage: $0 [cpu|gpu]" >&2; exit 2 ;; esac\n\nexport MPLCONFIGDIR=${MPLCONFIGDIR:-"$PWD/build/matplotlib"}\nmkdir -p results "$MPLCONFIGDIR"\narchive="proof-${mode}.zip"\nrm -f "$archive" results/comparison.csv\nrm -rf results/evidence\nexec > >(tee results/execution_log.txt) 2>&1\necho "K-means execution mode: $mode"\ndate -u\nuname -sm\ng++ --version\nif [[ $mode == gpu ]]; then\n  nvidia-smi\n  nvcc --version\n  # Force a CUDA rebuild if the GPU architecture changes between runs.\n  make -B cuda\n  binary=build/kmeans\n  backends=(cpu gpu)\nelse\n  make cpu\n  binary=build/kmeans_cpu\n  backends=(cpu)\nfi\n\npython3 scripts/generate_dataset.py\necho \'mode,points,k,iterations,repeats,median_ms,inertia\' > results/benchmark.csv\nfor size in 10000 50000 100000 500000 1000000; do\n  for backend in "${backends[@]}"; do\n    "$binary" "dataset/input_${size}.csv" 5 100 "$backend" "output/$backend/$size" 5 |\n      tee -a results/benchmark.csv\n  done\ndone\npython3 scripts/report.py "$mode"\n\nfor backend in "${backends[@]}"; do\n  mkdir -p "results/evidence/$backend"\n  cp "output/$backend/10000/"*.csv "results/evidence/$backend/"\n  cp "output/$backend/1000000/centroids.csv" "results/evidence/$backend/centroids_1000000.csv"\n  wc -l "output/$backend/1000000/clusters.csv"\ndone\ncp dataset/input_10000.csv results/evidence/input_10000.csv\nsha256sum dataset/input_*.csv > results/evidence/input_sha256.txt\necho "Finished $mode run and generated evidence."\n# Close the logging pipe and wait for tee to finish before archiving the log.\nexec 1>&- 2>&-\nwait "$!"\npython3 -c \'import shutil, sys; shutil.make_archive(sys.argv[1], "zip", ".", "results")\' "proof-${mode}"\n', 'requirements.txt': 'numpy==1.26.4\nmatplotlib==3.9.4\n', '.gitignore': 'build/\n.venv/\ndataset/\noutput/\nresults/\n*.zip\n__pycache__/\n', 'src/main.cc': '#include "kmeans.h"\n\n#include <algorithm>\n#include <chrono>\n#include <cmath>\n#include <filesystem>\n#include <fstream>\n#include <iomanip>\n#include <iostream>\n#include <limits>\n#include <sstream>\n#include <stdexcept>\n#include <string>\n#include <utility>\n\nbool Update(std::vector<Point>& centers, const std::vector<ClusterSum>& sums) {\n  double max_shift = 0;\n  for (size_t c = 0; c < centers.size(); ++c) {\n    const ClusterSum sum = sums[c];\n    if (sum.count == 0) continue;\n    const Point next{sum.x / sum.count, sum.y / sum.count};\n    max_shift = std::max(max_shift, std::hypot(next.x - centers[c].x,\n                                               next.y - centers[c].y));\n    centers[c] = next;\n  }\n  return max_shift <= 1e-6;\n}\n\nnamespace {\nstd::vector<Point> Read(const std::string& path) {\n  std::ifstream input(path);\n  std::string line;\n  if (!std::getline(input, line) || (line != "x,y" && line != "x,y\\r")) {\n    throw std::runtime_error("Expected CSV header x,y in " + path);\n  }\n  std::vector<Point> points;\n  while (std::getline(input, line)) {\n    std::istringstream row(line);\n    Point p{};\n    char comma;\n    if (!(row >> p.x >> comma >> p.y) || comma != \',\' ||\n        !(row >> std::ws).eof() || std::abs(p.x) > 1e100 || std::abs(p.y) > 1e100 ||\n        !std::isfinite(p.x) || !std::isfinite(p.y)) {\n      throw std::runtime_error("Invalid coordinate on line " + std::to_string(points.size() + 2));\n    }\n    if (points.size() == std::numeric_limits<int>::max()) {\n      throw std::runtime_error("Too many points");\n    }\n    points.push_back(p);\n  }\n  if (!input.eof()) throw std::runtime_error("Input read failed");\n  return points;\n}\n\nstd::vector<Point> Initialize(const std::vector<Point>& points, int k) {\n  std::vector<Point> centers{points.front()};\n  std::vector<double> nearest(points.size(), std::numeric_limits<double>::infinity());\n  while (centers.size() < static_cast<size_t>(k)) {\n    const Point last = centers.back();\n    for (size_t i = 0; i < points.size(); ++i) {\n      const double dx = points[i].x - last.x;\n      const double dy = points[i].y - last.y;\n      nearest[i] = std::min(nearest[i], dx * dx + dy * dy);\n    }\n    centers.push_back(points[std::max_element(nearest.begin(), nearest.end()) - nearest.begin()]);\n  }\n  return centers;\n}\n\nResult RunCpu(const std::vector<Point>& points, std::vector<Point> centers, int limit) {\n  const int k = centers.size();\n  std::vector<ClusterSum> sums(k);\n  std::vector<int> labels(points.size());\n  int iterations = 0;\n  while (iterations < limit) {\n    ++iterations;\n    std::fill(sums.begin(), sums.end(), ClusterSum{});\n    for (Point p : points) {\n      ClusterSum& sum = sums[Nearest(p, centers.data(), k)];\n      sum.x += p.x;\n      sum.y += p.y;\n      ++sum.count;\n    }\n    if (Update(centers, sums)) break;\n  }\n  for (size_t i = 0; i < points.size(); ++i) {\n    labels[i] = Nearest(points[i], centers.data(), k);\n  }\n  return {std::move(centers), std::move(labels), iterations};\n}\n\ndouble Inertia(const std::vector<Point>& points, const Result& result) {\n  double sum = 0;\n  for (size_t i = 0; i < points.size(); ++i) {\n    const Point center = result.centers[result.labels[i]];\n    const double dx = points[i].x - center.x;\n    const double dy = points[i].y - center.y;\n    sum += dx * dx + dy * dy;\n  }\n  return sum;\n}\n\nvoid Write(const std::filesystem::path& directory, const std::vector<Point>& points,\n           const Result& result) {\n  std::filesystem::create_directories(directory);\n  std::ofstream labels(directory / "clusters.csv");\n  std::ofstream centers(directory / "centroids.csv");\n  labels.exceptions(std::ios::failbit | std::ios::badbit);\n  centers.exceptions(std::ios::failbit | std::ios::badbit);\n  labels << std::setprecision(17) << "x,y,cluster\\n";\n  centers << std::setprecision(17) << "cluster,x,y\\n";\n  for (size_t i = 0; i < points.size(); ++i) {\n    labels << points[i].x << \',\' << points[i].y << \',\' << result.labels[i] << \'\\n\';\n  }\n  for (size_t c = 0; c < result.centers.size(); ++c) {\n    centers << c << \',\' << result.centers[c].x << \',\' << result.centers[c].y << \'\\n\';\n  }\n  labels.close();\n  centers.close();\n}\n\nint Integer(const std::string& text, int maximum) {\n  size_t end = 0;\n  const int value = std::stoi(text, &end);\n  if (end != text.size() || value < 1 || value > maximum) {\n    throw std::runtime_error("Integer argument outside supported range: " + text);\n  }\n  return value;\n}\n}  // namespace\n\nint main(int argc, char** argv) {\n  const char* usage = "Usage: kmeans INPUT.csv K MAX_ITERATIONS {cpu|gpu} OUTPUT_DIR [REPEATS]\\n";\n  try {\n    if (argc == 2 && std::string(argv[1]) == "--help") {\n      std::cout << usage;\n      return 0;\n    }\n    if (argc < 6 || argc > 7) throw std::runtime_error(usage);\n    const int k = Integer(argv[2], 256);\n    const int limit = Integer(argv[3], 10000);\n    const std::string mode = argv[4];\n    const int repeats = argc == 7 ? Integer(argv[6], 100) : 1;\n    if (mode != "cpu" && mode != "gpu") throw std::runtime_error(usage);\n#ifndef ENABLE_CUDA\n    if (mode == "gpu") throw std::runtime_error("Build with make cuda to use GPU mode");\n#endif\n    const auto points = Read(argv[1]);\n    if (points.size() < static_cast<size_t>(k)) throw std::runtime_error("K exceeds row count");\n    const auto initial = Initialize(points, k);\n    auto run = RunCpu;\n#ifdef ENABLE_CUDA\n    if (mode == "gpu") {\n      WarmGpu();\n      run = RunGpu;\n    }\n#endif\n    if (repeats > 1) run(points, initial, limit);\n    std::vector<double> times;\n    Result result{};\n    for (int trial = 0; trial < repeats; ++trial) {\n      const auto start = std::chrono::steady_clock::now();\n      result = run(points, initial, limit);\n      const auto end = std::chrono::steady_clock::now();\n      times.push_back(std::chrono::duration<double, std::milli>(end - start).count());\n    }\n    std::sort(times.begin(), times.end());\n    const double median = (times[(times.size() - 1) / 2] + times[times.size() / 2]) / 2;\n    Write(argv[5], points, result);\n    std::cout << std::setprecision(12) << mode << \',\' << points.size() << \',\'\n              << k << \',\' << result.iterations << \',\' << repeats << \',\' << median\n              << \',\' << Inertia(points, result) << \'\\n\';\n  } catch (const std::exception& error) {\n    std::cerr << error.what() << \'\\n\';\n    return 1;\n  }\n}\n', 'src/gpu.cu': '#include "kmeans.h"\n\n#include <cuda_runtime.h>\n#include <stdexcept>\n#include <string>\n#include <utility>\n\nnamespace {\nvoid Check(cudaError_t status, const char* operation) {\n  if (status != cudaSuccess) {\n    throw std::runtime_error(std::string(operation) + ": " + cudaGetErrorString(status));\n  }\n}\n\ntemplate <typename T>\nclass DeviceBuffer {\n public:\n  explicit DeviceBuffer(size_t size) : bytes_(size * sizeof(T)) {\n    Check(cudaMalloc(reinterpret_cast<void**>(&data), bytes_), "cudaMalloc");\n  }\n  ~DeviceBuffer() { cudaFree(data); }\n  DeviceBuffer(const DeviceBuffer&) = delete;\n  DeviceBuffer& operator=(const DeviceBuffer&) = delete;\n  void Upload(const T* src) {\n    Check(cudaMemcpy(data, src, bytes_, cudaMemcpyHostToDevice), "upload");\n  }\n  void Download(T* dst) const {\n    Check(cudaMemcpy(dst, data, bytes_, cudaMemcpyDeviceToHost), "download");\n  }\n  void Clear() { Check(cudaMemset(data, 0, bytes_), "clear"); }\n\n  T* data = nullptr;\n\n private:\n  size_t bytes_;\n};\n\n__global__ void Assign(const Point* points, int n, const Point* centers,\n                       int k, ClusterSum* totals, int* labels) {\n  extern __shared__ ClusterSum bins[];\n  if (totals) {\n    for (int c = threadIdx.x; c < k; c += blockDim.x) bins[c] = {};\n    __syncthreads();\n  }\n  const size_t i = size_t(blockIdx.x) * blockDim.x + threadIdx.x;\n  if (i < n) {\n    const Point point = points[i];\n    const int c = Nearest(point, centers, k);\n    if (totals) {\n      atomicAdd(&bins[c].x, point.x);\n      atomicAdd(&bins[c].y, point.y);\n      atomicAdd(&bins[c].count, 1);\n    } else {\n      labels[i] = c;\n    }\n  }\n  if (totals) {\n    __syncthreads();\n    for (int c = threadIdx.x; c < k; c += blockDim.x) {\n      if (bins[c].count == 0) continue;\n      atomicAdd(&totals[c].x, bins[c].x);\n      atomicAdd(&totals[c].y, bins[c].y);\n      atomicAdd(&totals[c].count, bins[c].count);\n    }\n  }\n}\n}  // namespace\n\nvoid WarmGpu() { Check(cudaFree(nullptr), "initialize CUDA"); }\n\nResult RunGpu(const std::vector<Point>& points, std::vector<Point> centers, int limit) {\n  const int n = points.size();\n  const int k = centers.size();\n  DeviceBuffer<Point> device_points(n);\n  DeviceBuffer<Point> device_centers(k);\n  DeviceBuffer<ClusterSum> device_sums(k);\n  DeviceBuffer<int> device_labels(n);\n  std::vector<ClusterSum> sums(k);\n  std::vector<int> labels(n);\n  device_points.Upload(points.data());\n  constexpr int threads = 256;\n  const int blocks = (size_t(n) + threads - 1) / threads;\n  int iterations = 0;\n  while (iterations < limit) {\n    ++iterations;\n    device_sums.Clear();\n    device_centers.Upload(centers.data());\n    Assign<<<blocks, threads, k * sizeof(ClusterSum)>>>(\n        device_points.data, n, device_centers.data, k, device_sums.data, nullptr);\n    Check(cudaGetLastError(), "launch assignment and reduction");\n    device_sums.Download(sums.data());\n    if (Update(centers, sums)) break;\n  }\n  device_centers.Upload(centers.data());\n  Assign<<<blocks, threads>>>(device_points.data, n, device_centers.data,\n                              k, nullptr, device_labels.data);\n  Check(cudaGetLastError(), "launch final assignment");\n  device_labels.Download(labels.data());\n  return {std::move(centers), std::move(labels), iterations};\n}\n', 'src/kmeans.h': '#ifndef KMEANS_H_\n#define KMEANS_H_\n#include <vector>\n\nstruct Point { double x, y; };\nstruct ClusterSum { double x, y; int count; };\nstruct Result {\n  std::vector<Point> centers;\n  std::vector<int> labels;\n  int iterations;\n};\n\n#ifdef __CUDACC__\n__host__ __device__\n#endif\ninline int Nearest(Point p, const Point* centers, int k) {\n  int best = 0;\n  double distance = 0;\n  for (int c = 0; c < k; ++c) {\n    const double dx = p.x - centers[c].x;\n    const double dy = p.y - centers[c].y;\n    const double squared = dx * dx + dy * dy;\n    if (c == 0 || squared < distance) {\n      distance = squared;\n      best = c;\n    }\n  }\n  return best;\n}\n\nbool Update(std::vector<Point>& centers, const std::vector<ClusterSum>& sums);\nResult RunGpu(const std::vector<Point>& points, std::vector<Point> centers, int limit);\nvoid WarmGpu();\n#endif\n', 'scripts/generate_dataset.py': 'from pathlib import Path\n\nimport numpy as np\n\nSIZES = (10_000, 50_000, 100_000, 500_000, 1_000_000)\nCENTERS = np.array([[-15, -15], [-15, 15], [0, 0], [15, -15], [15, 15]])\n\nif __name__ == "__main__":\n    random = np.random.RandomState(42)\n    Path("dataset").mkdir(exist_ok=True)\n    for size in SIZES:\n        labels = random.randint(0, len(CENTERS), size=size)\n        points = CENTERS[labels] + random.normal(0, 1.5, size=(size, 2))\n        path = Path("dataset") / f"input_{size}.csv"\n        np.savetxt(path, points, delimiter=",", fmt="%.6f", header="x,y", comments="")\n        print(f"Generated {path} with {size:,} points")\n', 'scripts/report.py': 'import csv\nimport sys\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\n\nfrom generate_dataset import SIZES\n\nmode = sys.argv[1]\nwith open("results/benchmark.csv") as stream:\n    rows = {(r["mode"], int(r["points"])): r for r in csv.DictReader(stream)}\n\nif mode == "gpu":\n    with open("results/comparison.csv", "w", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["points", "cpu_over_gpu", "label_disagreements",\n                         "max_centroid_distance", "relative_inertia_difference"])\n        for size in SIZES:\n            def read(backend, filename, **options):\n                return np.loadtxt(Path("output") / backend / str(size) / filename,\n                                  delimiter=",", skiprows=1, **options)\n\n            cpu = rows["cpu", size]\n            gpu = rows["gpu", size]\n            cpu_labels = read("cpu", "clusters.csv", usecols=2, dtype=int)\n            gpu_labels = read("gpu", "clusters.csv", usecols=2, dtype=int)\n            if cpu_labels.shape != (size,) or gpu_labels.shape != (size,):\n                raise ValueError(f"Incorrect row count for {size:,} points")\n            disagreements = np.count_nonzero(cpu_labels != gpu_labels)\n            distances = np.linalg.norm(\n                read("cpu", "centroids.csv", usecols=(1, 2)) -\n                read("gpu", "centroids.csv", usecols=(1, 2)), axis=1\n            )\n            inertia_error = abs(float(cpu["inertia"]) - float(gpu["inertia"]))\n            inertia_error /= max(1.0, float(cpu["inertia"]))\n            speedup = float(cpu["median_ms"]) / float(gpu["median_ms"])\n            writer.writerow([size, speedup, disagreements, max(distances), inertia_error])\n            if disagreements or max(distances) > 1e-6 or inertia_error > 1e-8:\n                raise ValueError(f"CPU and CUDA disagree on the seeded dataset of {size:,} points")\n    print("CPU and CUDA agree on labels, centroids and inertia at all five sizes")\n\npoints = np.loadtxt("output/" + mode + "/10000/clusters.csv", delimiter=",", skiprows=1)\ncenters = np.loadtxt("output/" + mode + "/10000/centroids.csv", delimiter=",", skiprows=1)\nfig, axes = plt.subplots(1, 2, figsize=(10, 5), layout="constrained", sharex=True, sharey=True)\nfor ax, title in zip(axes, ("Input points", f"{mode.upper()} cluster assignments")):\n    ax.set(title=title, xlabel="X", ylabel="Y", aspect="equal")\n    ax.grid(alpha=0.2)\naxes[0].scatter(points[:, 0], points[:, 1], s=2, alpha=0.35, color="#677886")\naxes[1].scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, c=points[:, 2], cmap="viridis")\naxes[1].scatter(centers[:, 1], centers[:, 2], marker="X", c="black",\n                edgecolors="white", s=100, label="Centroids")\naxes[1].legend(loc="lower center")\nfig.savefig("results/clusters.png", dpi=160)\nplt.close(fig)\n\nfig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")\nfor backend in ("cpu", "gpu") if mode == "gpu" else ("cpu",):\n    ax.plot(SIZES, [float(rows[backend, size]["median_ms"]) for size in SIZES],\n            marker="o", label=backend.upper())\nax.set(xscale="log", yscale="log", xlabel="Number of points",\n       ylabel="Median clustering time (ms)", title="K-means runtime by dataset size")\nax.grid(alpha=0.25, which="both")\nax.legend()\nfig.savefig("results/performance.png", dpi=160)\n'}
for name, source in files.items():
    path = project / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)


In [ ]:
hardware = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True).splitlines()[0].strip().replace(".", "")
subprocess.run(["bash", "run.sh", "gpu"], cwd=project, check=True,
               env={**__import__("os").environ, "CUDA_ARCH": "sm_" + hardware})


In [ ]:
from google.colab import files as colab_files
from IPython.display import Image, display

for name in ("clusters.png", "performance.png"):
    display(Image(filename=str(project / "results" / name)))
print((project / "results" / "comparison.csv").read_text())
colab_files.download(str(project / "proof-gpu.zip"))
